# Dia 4 — Duzentas conversas 💬

**Linux para OSINT — 21 dias** · baseado no curso de [cipher387](https://github.com/cipher387/linux-for-OSINT-21-day) (MIT)

📘 Texto completo do dia: [dia-04-processamento-em-lote](https://github.com/Ridd1kulusC0d3r/linux-para-osint-21-dias/tree/main/dias/dia-04-processamento-em-lote)

## 🎬 Missão

> As vítimas começaram a mandar as conversas exportadas do WhatsApp. Não
> uma, nem dez: **duzentas**, organizadas por dia em `desafio/conversas/`.
>
> — Lê tudo e me fala se o tal do mare_alta aparece — pede a Mestra Iara.
>
> Você abre a primeira. Depois a segunda. Na sétima, Bento cruza os braços:
> — Vai abrir as duzentas no bloco de notas? Eu faço em uma linha. Você
> também vai fazer.

**Sua tarefa:** descobrir em quais conversas o operador `mare_alta`
escorregou e montar a flag `VAZANTE{...}` com o que ele mandou "guardar",
na ordem em que as conversas aconteceram.

## ⚙️ Preparação

Rode a célula abaixo **uma vez**. Ela baixa o repositório para dentro do Colab e define a variável `L4O_HOME`, que todas as outras células usam.

> No Colab você é `root` e o `apt` funciona. Cada célula `%%bash` roda num processo separado: variáveis e `cd` não passam de uma célula para outra.

In [ ]:
import os, subprocess
from pathlib import Path

# Se já estamos dentro do repositório (Codespaces, máquina local), usa ele.
aqui = Path.cwd()
candidatos = [aqui, *aqui.parents]
raiz = next((p for p in candidatos if (p / 'lib' / 'comum.sh').exists()), None)

if raiz is None:  # Colab: clona o repositório
    raiz = Path('/content/linux-para-osint-21-dias')
    if not raiz.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Ridd1kulusC0d3r/linux-para-osint-21-dias', str(raiz)], check=True)

os.environ['L4O_HOME'] = str(raiz)
os.environ['L4O_CORES'] = '1'
print('Repositório em', raiz)

### Quanta coisa tem aqui?

```text
Antes de processar em lote, meça o lote. `find` lista arquivos em qualquer
profundidade; `wc -l` conta as linhas da saída (= quantos arquivos).
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-04-processamento-em-lote" && source "$L4O_HOME/lib/comum.sh"
titulo "Tamanho do lote"
ls desafio/conversas
echo "Arquivos .txt: $(find desafio/conversas -name '*.txt' -type f | wc -l)"
du -sh desafio/conversas          # espaço ocupado

### find: achando arquivos por critério

```text
  -name '*.txt'   nome bate com o padrão (use aspas!)
  -type f         só arquivos (d = só pastas)
  -newer / -mtime / -size   filtram por data e tamanho
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-04-processamento-em-lote" && source "$L4O_HOME/lib/comum.sh"
titulo "find com critérios"
find desafio/conversas -name "*.txt" -type f | head -n 5
echo "..."
echo "--- só as conversas do dia 2026-03-05:"
find desafio/conversas -path '*2026-03-05*' -name '*.txt' | sort | head -n 3
echo "--- conversas com mais de 400 bytes:"
find desafio/conversas -type f -size +400c | wc -l

### Laço for sobre o resultado do find (o script original)

```text
O script original fazia:  for i in `find txt_files/. -name "*.txt"`; do echo "$i"; done
Funciona, mas quebra se um nome tiver espaço. A forma segura é
`find ... -print0 | while IFS= read -r -d '' arquivo` (separa por caractere nulo).
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-04-processamento-em-lote" && source "$L4O_HOME/lib/comum.sh"
titulo "Imprimindo nomes com um laço"
find desafio/conversas/2026-03-01 -name "*.txt" -type f -print0 | sort -z |
  while IFS= read -r -d '' arquivo; do
    echo "$arquivo"
  done | head -n 5

### Laço com contador: quantas linhas tem cada conversa?

```text
Dentro do laço você pode fazer qualquer coisa com cada arquivo.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-04-processamento-em-lote" && source "$L4O_HOME/lib/comum.sh"
titulo "Linhas por conversa (amostra)"
for arquivo in desafio/conversas/2026-03-01/conversa-00{1..5}.txt; do
  printf '%-45s %s linhas\n' "$arquivo" "$(wc -l < "$arquivo")"
done

### Conversão em massa: .txt -> .csv

```text
O original trocava espaços por vírgulas com sed e salvava como .csv.
Nossas conversas usam " | " como separador; trocamos por ";" (CSV brasileiro).
"${arquivo%.txt}" remove o final ".txt" do nome — e acrescentamos ".csv".
Trabalhamos numa CÓPIA (treino/) para não mexer na evidência.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-04-processamento-em-lote" && source "$L4O_HOME/lib/comum.sh"
titulo "Convertendo um dia inteiro para CSV"
mkdir -p treino
cp -r desafio/conversas/2026-03-01 treino/
for arquivo in treino/2026-03-01/*.txt; do
  sed 's/ | /;/g' "$arquivo" > "${arquivo%.txt}.csv"
done
ls treino/2026-03-01 | head -n 4
head -n 2 treino/2026-03-01/conversa-001.csv
rm -r treino

### xargs: transformando uma lista em argumentos

```text
`xargs` pega a lista que chega pela entrada e passa como argumentos para
outro comando. É o "faça isso com cada um" mais rápido do Linux.
  find ... | xargs wc -l        → wc recebe todos os arquivos de uma vez
  -0  (com find -print0)       → aguenta nomes com espaço
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-04-processamento-em-lote" && source "$L4O_HOME/lib/comum.sh"
titulo "find + xargs"
find desafio/conversas/2026-03-01 -name '*.txt' -print0 | xargs -0 wc -l | tail -n 1
echo "--- quem mais aparece como atendente em TODAS as conversas:"
find desafio/conversas -name '*.txt' -print0 | xargs -0 cut -d'|' -f2 | sort | uniq -c | sort -rn | head -n 6

### xargs com grep -l: QUAIS arquivos contêm uma palavra?

```text
grep -l imprime só o NOME dos arquivos que têm a palavra (não as linhas).
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-04-processamento-em-lote" && source "$L4O_HOME/lib/comum.sh"
titulo "Arquivos que citam 'cashback'"
find desafio/conversas -name '*.txt' -print0 | xargs -0 grep -l 'cashback' | wc -l

### Apagando em lote (com cuidado!)

```text
O original apagava zips esquecidos:  find txt_files -name "*.zip" | xargs rm -f
Antes de apagar, SEMPRE rode só o find para ver a lista.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-04-processamento-em-lote" && source "$L4O_HOME/lib/comum.sh"
titulo "Apagando arquivos temporários em lote"
mkdir -p treino
touch treino/a.zip treino/b.zip treino/manter.txt
echo "Vou apagar:"; find treino -name "*.zip" -type f
find treino -name "*.zip" -type f -print0 | xargs -0 rm -f
echo "Sobrou:"; ls treino
rm -r treino

### xargs -I: colocar cada item num lugar específico do comando

```text
Com -I NOME, o xargs troca NOME por cada linha da entrada.
O original zipava os arquivos listados em file_names.txt um a um.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-04-processamento-em-lote" && source "$L4O_HOME/lib/comum.sh"
titulo "Zipando só as conversas selecionadas"
precisa zip
mkdir -p treino
xargs -I nomearq zip -q treino/selecionadas desafio/conversas/nomearq < desafio/selecionadas.txt
unzip -l treino/selecionadas.zip
rm -r treino

### xargs -P: em paralelo

```text
-P 4 roda até 4 processos ao mesmo tempo; -n 1 passa um arquivo por vez.
Em milhares de arquivos (ou downloads), isso corta o tempo pela metade ou mais.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-04-processamento-em-lote" && source "$L4O_HOME/lib/comum.sh"
titulo "Contando palavras em paralelo"
find desafio/conversas -name '*.txt' -print0 | xargs -0 -n 20 -P 4 cat | wc -w

### find -exec: a alternativa ao xargs

```text
`-exec comando {} +` faz o mesmo que xargs, sem precisar de pipe.
{} é o lugar do nome do arquivo; o + junta vários arquivos numa chamada.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-04-processamento-em-lote" && source "$L4O_HOME/lib/comum.sh"
titulo "find -exec"
find desafio/conversas -name 'conversa-00[1-3].txt' -exec wc -l {} +

### Pista: o operador apareceu?

```text
O nome do operador d'Os Vazantes é mare_alta. Será que ele escorregou?
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-04-processamento-em-lote" && source "$L4O_HOME/lib/comum.sh"
titulo "Procurando o operador"
find desafio/conversas -name '*.txt' -print0 | xargs -0 grep -l 'mare_alta' | wc -l
dica "Alguns arquivos citam mare_alta. Ordene e leia o que ele mandou guardar."

## 🐍 Ponte Python

O mesmo resultado só com a biblioteca padrão do Python:

In [ ]:
!cd "$L4O_HOME/dias/dia-04-processamento-em-lote" && python3 equivalente.py

## 🧩 Desafio

Encontrou a flag? Troque o texto abaixo e rode a célula.

Dicas e passo a passo estão no README do dia.

In [ ]:
flag = 'VAZANTE{troque_aqui}'  # ✏️ cole sua resposta
!cd "$L4O_HOME/dias/dia-04-processamento-em-lote" && bash checar.sh "{flag}"

---

📝 Anote o que aprendeu no `DIARIO.md` do repositório.

Feito por [Deivison Lourenço](https://github.com/Ridd1kulusC0d3r) · curso original de [cipher387 / Cyber Detective](https://cybdetective.com)